In [1]:
!pip install pyspark -q

import pyspark
print("PySpark version:", pyspark.__version__)

from google.colab import files
uploaded = files.upload()

PySpark version: 4.0.2


Saving patients_data_with_alerts.xlsx to patients_data_with_alerts.xlsx


In [2]:
import pandas as pd

df = pd.read_excel('patients_data_with_alerts.xlsx')
print("Source records:", df.shape)
df.head()

Source records: (50000, 13)


,Patient Number,Heart Rate (bpm),SpO2 Level (%),Systolic Blood Pressure (mmHg),Diastolic Blood Pressure (mmHg),Body Temperature (°C),Fall Detection,Predicted Disease,Data Accuracy (%),Heart Rate Alert,SpO2 Level Alert,Blood Pressure Alert,Temperature Alert
0,1,96,92,101,89,36.904680,No,Diabetes Mellitus,95,Normal,Normal,Normal,Normal
1,2,76,83,103,85,37.254129,Yes,Asthma,91,Normal,Low,Normal,Abnormal
2,3,92,88,123,70,36.539418,Yes,Asthma,86,Normal,Low,Normal,Normal
3,4,137,84,167,97,37.018767,Yes,Asthma,99,High,Low,High,Normal
4,5,76,81,175,80,37.328671,No,Normal,93,Normal,Low,High,Abnormal


In [4]:
import random
from datetime import datetime, timedelta

random.seed(42)

NUM_PATIENTS = 40
READINGS_PER_PATIENT = 240
INTERVAL_SEC = 30
HR_THRESHOLD = 100

sample = df.sample(n=NUM_PATIENTS, random_state=42).reset_index(drop=True)
start_time = datetime(2026, 6, 13, 8, 0, 0)

rows = []
for new_id, (_, r) in enumerate(sample.iterrows(), start=1):
    base_hr, base_spo2 = r['Heart Rate (bpm)'], r['SpO2 Level (%)']
    base_sys, base_dia = r['Systolic Blood Pressure (mmHg)'], r['Diastolic Blood Pressure (mmHg)']
    base_temp = r['Body Temperature (°C)']

    for i in range(READINGS_PER_PATIENT):
        ts = start_time + timedelta(seconds=i * INTERVAL_SEC)
        hr = base_hr + random.randint(-5, 5)

        if new_id % 4 == 0 and 60 <= i <= 150:
            hr = random.randint(102, 130)
        hr = max(40, min(180, hr))

        rows.append([
            ts.strftime('%Y-%m-%d %H:%M:%S'),
            new_id,
            hr,
            max(70, min(100, base_spo2 + random.randint(-2, 2))),
            max(80, min(200, base_sys + random.randint(-5, 5))),
            max(50, min(130, base_dia + random.randint(-5, 5))),
            round(base_temp + random.uniform(-0.3, 0.3), 2)
        ])

stream_df = pd.DataFrame(rows, columns=[
    'timestamp', 'patient_id', 'heart_rate', 'spo2', 'systolic_bp', 'diastolic_bp', 'body_temp'
])
print("Generated readings:", stream_df.shape)
stream_df.head()

Generated readings: (9600, 7)


,timestamp,patient_id,heart_rate,spo2,systolic_bp,diastolic_bp,body_temp
0,2026-06-13 08:00:00,1,108,95,157,71,37.82
1,2026-06-13 08:00:30,1,100,95,167,75,37.73
2,2026-06-13 08:01:00,1,104,95,157,68,37.80
3,2026-06-13 08:01:30,1,106,99,157,75,37.79
4,2026-06-13 08:02:00,1,108,99,163,70,37.94


In [5]:
import os, shutil

CHUNK_SIZE = 200


shutil.rmtree('chunks', ignore_errors=True)
shutil.rmtree('stream_source', ignore_errors=True)
shutil.rmtree('checkpoint', ignore_errors=True)
os.makedirs('chunks', exist_ok=True)
os.makedirs('stream_source', exist_ok=True)

for i in range(0, len(stream_df), CHUNK_SIZE):
    chunk = stream_df.iloc[i:i+CHUNK_SIZE]
    chunk.to_csv(f'chunks/batch_{i//CHUNK_SIZE:04d}.csv', index=False)

print("Chunk files created:", len(os.listdir('chunks')))

Chunk files created: 48


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import window, avg, col

spark = SparkSession.builder \
    .appName("ICU_PatientVitals_Streaming") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("patient_id", IntegerType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("spo2", IntegerType(), True),
    StructField("systolic_bp", IntegerType(), True),
    StructField("diastolic_bp", IntegerType(), True),
    StructField("body_temp", DoubleType(), True),
])

print("Spark session ready:", spark.version)

Spark session ready: 4.0.2


In [7]:
HR_THRESHOLD = 100
WINDOW_DURATION = "2 minutes"
WATERMARK_DELAY = "1 minute"


raw_stream = spark.readStream \
    .option("header", "true") \
    .schema(schema) \
    .csv("stream_source/")


windowed_hr = raw_stream \
    .withWatermark("timestamp", WATERMARK_DELAY) \
    .groupBy(
        window(col("timestamp"), WINDOW_DURATION),
        col("patient_id")
    ) \
    .agg(avg("heart_rate").alias("avg_hr"))


alerts = windowed_hr.filter(col("avg_hr") > HR_THRESHOLD)

print("Pipeline defined: readStream -> tumbling window -> alert filter")

Pipeline defined: readStream -> tumbling window -> alert filter


In [8]:
def process_batch(batch_df, batch_id):
    alert_rows = batch_df.collect()
    if not alert_rows:
        return

    print("=" * 70)
    print(f"ICU HEART RATE MONITORING — CLINICAL ALERTS  (Batch {batch_id})")
    print(f"Scenario B: Tumbling 2-min Window | Threshold: avg HR > {HR_THRESHOLD} bpm")
    print("=" * 70)
    print(f"{'patient_id':<12}{'window_start':<22}{'window_end':<22}{'avg_hr':<10}{'status'}")
    print("-" * 70)
    for r in sorted(alert_rows, key=lambda x: (x['patient_id'], x['window']['start'])):
        ws = r['window']['start'].strftime('%Y-%m-%d %H:%M:%S')
        we = r['window']['end'].strftime('%Y-%m-%d %H:%M:%S')
        print(f"{r['patient_id']:<12}{ws:<22}{we:<22}{r['avg_hr']:<10.1f}CLINICAL ALERT")
    print("=" * 70)
    print(f"Alerts in this batch: {len(alert_rows)}")
    print("=" * 70)
    print()

print("Output sink function defined.")

Output sink function defined.


In [9]:
query = alerts.writeStream \
    .outputMode("update") \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", "checkpoint/") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Streaming query started.")
import time
time.sleep(10)
print("Ready to receive data.")

Streaming query started.
Ready to receive data.


In [10]:
import shutil, time

chunk_files = sorted(os.listdir('chunks'))

for i, f in enumerate(chunk_files):
    shutil.copy(f'chunks/{f}', f'stream_source/{f}')
    if i % 5 == 0:
        time.sleep(6)
    else:
        time.sleep(1)

print("All sensor data streamed.")
time.sleep(15)
print("Stream complete.")

ICU HEART RATE MONITORING — CLINICAL ALERTS  (Batch 0)
Scenario B: Tumbling 2-min Window | Threshold: avg HR > 100 bpm
patient_id  window_start          window_end            avg_hr    status
----------------------------------------------------------------------
1           2026-06-13 08:00:00   2026-06-13 08:02:00   104.5     CLINICAL ALERT
1           2026-06-13 08:02:00   2026-06-13 08:04:00   102.2     CLINICAL ALERT
1           2026-06-13 08:04:00   2026-06-13 08:06:00   101.8     CLINICAL ALERT
1           2026-06-13 08:06:00   2026-06-13 08:08:00   101.2     CLINICAL ALERT
1           2026-06-13 08:08:00   2026-06-13 08:10:00   102.5     CLINICAL ALERT
1           2026-06-13 08:10:00   2026-06-13 08:12:00   101.5     CLINICAL ALERT
1           2026-06-13 08:12:00   2026-06-13 08:14:00   103.8     CLINICAL ALERT
1           2026-06-13 08:14:00   2026-06-13 08:16:00   104.2     CLINICAL ALERT
1           2026-06-13 08:16:00   2026-06-13 08:18:00   103.0     CLINICAL ALERT
1       